In [10]:
import glob
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

In [7]:
def load_day_night_dataset(base_dir='images'):
  images = []
  labels = []

  for root, dirs, files in os.walk(base_dir):
    folder_name = os.path.basename(root).lower()

    if folder_name in ['day', 'night']:
      label = 1 if folder_name == 'day' else 0

      for file_name in files:
        if file_name.lower().endswith(
            ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
        ):
          file_path = os.path.join(root, file_name)
          img = cv2.imread(file_path)
          if img is not None:
            # OpenCV loads in BGR; convert to RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(label)

  return images, np.array(labels)

images, y = load_day_night_dataset('images')

print(f'Total images successfully loaded: {len(images)}')
print(f'Daytime images  (Class 1): {np.sum(y == 1)}')
print(f'Nighttime images (Class 0): {np.sum(y == 0)}')

Total images successfully loaded: 400
Daytime images  (Class 1): 200
Nighttime images (Class 0): 200


In [8]:
def extract_color_histograms(images, bins_per_channel=32):
  features = []
  for img in images:
    resized = cv2.resize(img, (600, 400))
    hsv = cv2.cvtColor(resized, cv2.COLOR_RGB2HSV)

    ranges = [[0, 180], [0, 256], [0, 256]]
    hist_combined = []

    for c in range(3):
      hist = cv2.calcHist(
          [hsv], [c], None, [bins_per_channel], ranges[c]
      ).flatten()
      if hist.sum() > 0:
        hist = hist / hist.sum()
      hist_combined.extend(hist)

    features.append(hist_combined)

  return np.array(features)

X = extract_color_histograms(images, bins_per_channel=32)
print(f'Feature matrix shape (X): {X.shape}')
print(f'Target vector shape  (y): {y.shape}')

Feature matrix shape (X): (400, 96)
Target vector shape  (y): (400,)


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set size (80%): {X_train_scaled.shape[0]} samples')
print(f'Testing set size  (20%): {X_test_scaled.shape[0]} samples')

Training set size (80%): 320 samples
Testing set size  (20%): 80 samples


In [11]:
base_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
base_rbf.fit(X_train_scaled, y_train)
base_train_acc = accuracy_score(y_train, base_rbf.predict(X_train_scaled))
base_test_acc = accuracy_score(y_test, base_rbf.predict(X_test_scaled))

print(f'Baseline RBF Train Accuracy: {base_train_acc:.4f}')
print(f'Baseline RBF Test Accuracy : {base_test_acc:.4f}\n')

param_grid = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'gamma': ['scale', 'auto', 0.1, 0.01, 0.001],
}

grid = GridSearchCV(
    estimator=SVC(kernel='rbf', random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=True,
    n_jobs=-1,
)
grid.fit(X_train_scaled, y_train)

print(f'Optimal Parameters: {grid.best_params_}')
print(f'Best 5-Fold Cross-Validation Accuracy: {grid.best_score_:.4f}')

Baseline RBF Train Accuracy: 1.0000
Baseline RBF Test Accuracy : 1.0000

Optimal Parameters: {'C': 100.0, 'gamma': 0.001}
Best 5-Fold Cross-Validation Accuracy: 1.0000


In [12]:
results_df = pd.DataFrame(grid.cv_results_)

summary_table = pd.DataFrame({
    'C': results_df['param_C'],
    'gamma': results_df['param_gamma'],
    'Mean CV Train Accuracy': results_df['mean_train_score'].round(4),
    'Mean CV Validation Accuracy': results_df['mean_test_score'].round(4),
})

summary_table = summary_table.sort_values(
    by='Mean CV Validation Accuracy', ascending=False
).reset_index(drop=True)
print(summary_table.to_string())

best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)
final_test_acc = accuracy_score(y_test, y_pred_best)

print('\nFinal Evaluation on Test Set (80:20)')
print(f'Final Test Accuracy: {final_test_acc:.4f} ({final_test_acc * 100:.2f}%)')
print('\nClassification Report:')
print(
    classification_report(
        y_test, y_pred_best, target_names=['Night (0)', 'Day (1)']
    )
)
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_best))

        C  gamma  Mean CV Train Accuracy  Mean CV Validation Accuracy
0   100.0  0.001                  1.0000                       1.0000
1    10.0   auto                  1.0000                       0.9969
2    10.0  scale                  1.0000                       0.9969
3   100.0   auto                  1.0000                       0.9969
4    10.0  0.001                  1.0000                       0.9969
5   100.0  scale                  1.0000                       0.9969
6    10.0   0.01                  1.0000                       0.9969
7   100.0   0.01                  1.0000                       0.9969
8     1.0   auto                  1.0000                       0.9906
9     1.0  scale                  1.0000                       0.9906
10    1.0   0.01                  1.0000                       0.9906
11    1.0  0.001                  0.9961                       0.9875
12   10.0    0.1                  1.0000                       0.9531
13  100.0    0.1    

Color histograms in the HSV color space summarize brightness variations across the entire scene. Daytime images produce high bin counts between index 20–31 (values 160–255), while nighttime images cluster heavily in bins 0–8 (values 0–65).  Lower gamma values allow the RBF kernel to generate a smooth, well-generalized boundary. Excessive gamma values isolate individual training instances. Bins for background regions can have disproportionately large pixel counts compared to high-frequency color accents. Applying StandardScaler prevents any single bin from dominating Euclidean distance calculations during SVM margin optimization